# 05 — Analyses added in the revision

The reviewers of the ML4H 2026 submission asked for a set of analyses that the
first version did not have. This notebook runs all of them, in order, from what
`03_train` and `04_results` left on disk. Each step is a script in
`src/revision/`, so it can also be run on its own (`python src/revision/<name>.py`
from the repository root); the scripts carry the detailed documentation.

**Inputs.** `data/train/predictions/`, `data/train/models/` and
`data/train/tables/` (from `03` and `04`), and `data/master/`.
**Outputs.** CSV and parquet files in `data/train/revision/`, read by `06_paper_outputs`.

**Time convention.** A forecast computed from row *t* (cases up to *t-1*,
weather up to *t*) is issued at the start of month *t+2*, when the cases of *t-1*
have had about 60 days to consolidate. Horizon *h* = 1 is then a nowcast of a
month that has already ended, *h* = 2 forecasts the month that begins on the
issue date, and *h* = 3 is the first horizon with a full month of warning. The
effective lead time is *h* − 2 months.

**Two reference months for "calm".** Onset and continuation are defined by the
municipality's outbreak state in a reference month: *t*, the month the row reads
(the reference of the submitted version, not yet consolidated on the issue date),
or *t-1*, the last month that is consolidated on the issue date (real time).

**Uncertainty.** 95% intervals come from a bootstrap over municipalities (1,000
replicates, fixed seed), which captures variation between municipalities. The
range across test years is reported beside it, because it is usually wider.

| Step | Script | Retrains | Approx. time |
|---|---|---|---|
| Checks | `comum.checar_metricas`, `treino.conferir_reproducao` | one fold | 2 min |
| A3 | `a3_tempo_real.py`: onsets that started before the issue date; which inputs are published by then | no | 1 min |
| C1 | `c1_onset_continuacao.py`: onset and continuation, h = 1 to 6, both references, with CIs | no | 2 min |
| A5 | `a5_incerteza.py`: Tables 1 and 2 with CIs | no | 1 min |
| A6 | `a6_orcamento_calmos.py`: fixed alert budget among calm municipalities, and the alerts-versus-onsets curve | no | 3 min |
| A9 | `a9_shap_onset.py`: SHAP by feature family on onset rows | no | 5 min |
| B1 | `b1_dobra_2025.py`: 2025 as a fourth test year, h = 1 to 6 | 2025 fold | 15 min |
| B2 | `b2_modelo_onset.py`: a model trained only on calm months | yes | 10 min |
| B3 | `b3_limiares_passado.py`: relative-definition thresholds from past years only | yes | 15 min |
| B4 | `b4_tuning.py`: random hyperparameter search, chosen on the year before each test year | yes | 45 min |
| C2 | `c2_calibracao.py`: isotonic calibration fitted on the year before each test year | 2021 fold | 5 min |

In [1]:
import runpy
import sys
import time
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
REVISION = PROJECT_ROOT / "src" / "revision"
sys.path.insert(0, str(REVISION))
OUT = PROJECT_ROOT / "data" / "train" / "revision"

def run(script):
    """Run one analysis script as if from the command line, and time it."""
    t0 = time.time()
    runpy.run_path(str(REVISION / script), run_name="__main__")
    print(f"\n[{script}: {time.time() - t0:.0f}s]")

## Checks before anything else

The weighted metrics used for the bootstrap must match scikit-learn, including
when the highest-scored row belongs to a municipality a replicate did not draw.
Retraining the 2024 fold of dengue/inc100/h = 1 must reproduce the predictions
that `03` saved, which is what lets the retraining analyses below reuse the
saved 2022 to 2024 predictions as their reference.

In [2]:
import comum
import treino

comum.checar_metricas()
print("weighted AUROC and AP match scikit-learn")
treino.conferir_reproducao();

Merged: (89010, 99)


Features: (89010, 128)
All-NaN columns: none
Object columns remaining: none
dtypes sample: {'mgrd01_seca': dtype('float32'), 'msau28_pacs': dtype('float32'), 'mmam2612_moradia_risco': dtype('float32')}
dengue inc100 h=1: 88,365 non-null, 14,116 outbreaks
h=1:
  test 2022: train=53,535  test=7,740
  test 2023: train=61,275  test=7,740
  test 2024: train=69,015  test=7,740
h=6:


  test 2022: train=50,310  test=7,740
  test 2023: train=58,050  test=7,740
  test 2024: train=65,790  test=7,740
weighted AUROC and AP match scikit-learn


diferença máxima contra as predições do 03: {'rf': 3.3306690738754696e-16, 'xgb': 0.0}


## Part A — Evaluation by issue date, without retraining

**A3.** How many real-time onsets (calm at *t-1*, outbreak at *t+h*) had in fact
already begun before the issue date, and which inputs are not yet published on
that date.

In [3]:
run("a3_tempo_real.py")

 h               alvo_na_emissao  n_onset_ref_tm1  ja_em_surto_em_t  ja_em_surto_em_t1_nao_t  pct_ja_comecou  n_onset_limpo  n_continuacao_ref_tm1  n_onset_ref_t_paper
 1              mes ja encerrado             2671              1249                        0           0.468           1422                   3212                 1513
 2     mes que comeca na emissao             3653              1080                     1199           0.624           1374                   2230                 2671
 3 1 mes(es) inteiro(s) a frente             4412               886                     1033           0.435           2493                   1471                 3653
 4 2 mes(es) inteiro(s) a frente             4920               630                      853           0.301           3437                    963                 4412
 5 3 mes(es) inteiro(s) a frente             5156               395                      597           0.192           4164                    727              

**C1.** Base rate, recall and precision at 0.5, AUROC and AUPRC, separately for
onsets and continuations, at every horizon and with both reference months.

In [4]:
run("c1_onset_continuacao.py")

h=1 ok


h=2 ok


h=3 ok


h=4 ok


h=5 ok


h=6 ok


               auprc               auroc           base_rate           precision              recall       
subset  continuation  onset continuation  onset continuation  onset continuation  onset continuation  onset
h model                                                                                                    
1 rf           0.915  0.628        0.906  0.887        0.563  0.152        0.778  0.557        0.915  0.624
  xgb          0.923  0.646        0.909  0.896        0.563  0.152        0.829  0.592        0.860  0.576
2 rf           0.846  0.621        0.902  0.861        0.395  0.208        0.753  0.584        0.806  0.627
  xgb          0.853  0.642        0.902  0.872        0.395  0.208        0.788  0.591        0.722  0.612
3 rf           0.747  0.612        0.888  0.834        0.263  0.250        0.731  0.592        0.650  0.651
  xgb          0.765  0.654        0.890  0.853        0.263  0.250        0.769  0.608        0.571  0.628
4 rf           0.633  0.605 

**A5.** The submitted version's Tables 1 and 2 (dengue, inc100, *h* = 1,
reference *t*) with 95% intervals. The cell after it checks that these
recomputations, which use their own code, agree with the tables `04` wrote.

In [5]:
run("a5_incerteza.py")

Tabela 1 com incerteza
  Persistence (month t)                  AUROC 0.825 [0.820, 0.829] dobras 0.820-0.831 | AUPRC 0.612 [0.601, 0.623] dobras 0.560-0.700
  Persistence (real time, month t-1)     AUROC 0.690 [0.682, 0.697] dobras 0.684-0.695 | AUPRC 0.404 [0.391, 0.418] dobras 0.324-0.531
  Climatology (month t+h)                AUROC 0.778 [0.768, 0.787] dobras 0.686-0.829 | AUPRC 0.515 [0.495, 0.536] dobras 0.459-0.550
  Logistic regression                    AUROC 0.858 [0.851, 0.866] dobras 0.815-0.891 | AUPRC 0.663 [0.648, 0.678] dobras 0.581-0.727
  Random forest                          AUROC 0.914 [0.909, 0.919] dobras 0.879-0.938 | AUPRC 0.806 [0.795, 0.818] dobras 0.796-0.818
  XGBoost                                AUROC 0.921 [0.916, 0.925] dobras 0.892-0.944 | AUPRC 0.820 [0.808, 0.831] dobras 0.797-0.853
  LightGBM                               AUROC 0.919 [0.913, 0.924] dobras 0.889-0.943 | AUPRC 0.819 [0.807, 0.830] dobras 0.799-0.849

Tabela 2 com incerteza (%)
  Pe

In [6]:
# Two independent code paths must give the same headline numbers.
t04 = pd.read_csv(PROJECT_ROOT / "data" / "train" / "tables" / "table3_global_dengue_inc100.csv", index_col=0)
a5 = pd.read_csv(OUT / "a5_tabela1_ic.csv").set_index("model")
for model, label in [("rf", "Random Forest"), ("xgb", "XGBoost"), ("lgbm", "LightGBM"),
                     ("logreg", "Logistic regression"), ("persistence", "Persistence (month t)"),
                     ("persistence_tm1", "Persistence (real time)"), ("climatology_corr", "Climatology")]:
    for metric in ("auroc", "auprc"):
        assert abs(round(a5.loc[model, metric], 3) - t04.loc[label, metric]) < 1e-9, (model, metric)
onset04 = pd.read_csv(PROJECT_ROOT / "data" / "train" / "tables" / "table_onset_vs_continuation.csv", index_col=0)
c1 = pd.read_csv(OUT / "c1_onset_continuacao.csv")
for model, label in [("rf", "Random Forest"), ("xgb", "XGBoost"), ("climatology_corr", "Climatology")]:
    for subset, col in (("onset", "recall_onset"), ("continuation", "recall_continuation")):
        v = c1[(c1.h == 1) & (c1.ref == "t") & (c1.subset == subset) & (c1.model == model)]["recall"].item()
        assert abs(v - onset04.loc[label, col]) < 1e-12, (model, subset)
print("04 and the revision scripts agree on Tables 1 and 2")

04 and the revision scripts agree on Tables 1 and 2


**A6.** Each month, rank only the municipalities that were calm in the last
consolidated month and support the top *k*, or a fixed share of them. The
proportional budgets, from 1% to 100% of calm municipalities, trace the
trade-off between alerts raised and onsets caught.

In [7]:
run("a6_orcamento_calmos.py")


=== h = 1 (emissão no início de t+2), taxa de onset entre calmos 15.2% ===
budget                              k=10  k=25  k=50  1% dos calmos  2% dos calmos  5% dos calmos  10% dos calmos  15% dos calmos  20% dos calmos  30% dos calmos  40% dos calmos  50% dos calmos  75% dos calmos  100% dos calmos
model                                                                                                                                                                                                                             
Random forest                       46.9  43.4  39.6           53.5           47.7           45.5            40.7            37.1            33.9            29.1            25.4            22.9            18.3             15.2
XGBoost                             48.1  45.1  39.8           49.4           50.3           45.5            41.1            37.7            34.7            29.7            26.1            23.4            18.5             15.2
LightGBM        

**A9.** SHAP attributions by feature family on onset rows, against the pooled
sample and continuation rows (random forest, dengue, inc100, *h* = 1).

In [8]:
run("a9_shap_onset.py")

Merged: (89010, 99)


Features: (89010, 128)
All-NaN columns: none
Object columns remaining: none
dtypes sample: {'mgrd01_seca': dtype('float32'), 'msau28_pacs': dtype('float32'), 'mmam2612_moradia_risco': dtype('float32')}
dengue inc100 h=1: 88,365 non-null, 14,116 outbreaks
h=1:
  test 2022: train=53,535  test=7,740
  test 2023: train=61,275  test=7,740
  test 2024: train=69,015  test=7,740
h=6:


  test 2022: train=50,310  test=7,740
  test 2023: train=58,050  test=7,740
  test 2024: train=65,790  test=7,740
125 features in 10 groups, none left over

                   features      role
other arboviruses        46    tested
climate                  20  baseline
target history           16  baseline
environment              13    tested
socioeconomic            12    tested
health services          10    tested
sanitation                4    tested
seasonality               2  baseline
geography                 1    tested
mobility                  1    tested


2022 aggregate (sample 1500)   1500 linhas [50s]


2022 onset (ref t)              443 linhas [15s]


2022 continuation (ref t)      1132 linhas [38s]


... (6 outputs left out)

2024 onset (ref t)              709 linhas [37s]


2024 continuation (ref t)      2310 linhas [121s]


2024 onset (ref t-1)           1240 linhas [65s]


subset             aggregate (sample 1500)  onset (ref t)  onset (ref t-1)  continuation (ref t)
group                                                                                           
climate                               43.5           40.7             40.6                  35.8
target history                        39.9           41.6             41.9                  50.0
seasonality                            9.1            8.5              8.6                   7.5
health services                        2.9            4.0              3.8                   2.6
environment                            2.0            2.2              2.1                   1.7
socioeconomic                          1.5            1.7              1.7                   1.3
sanitation                             0.5            0.6              0.5                   0.4
mobility                               0.2            0.2              0.2                   0.2
geography                     

## Part B — Analyses that retrain models

All use the configuration of `03_train` (same features, hyperparameters and
expanding-window folds) unless the analysis is precisely about changing one of
them.

**B1.** 2025 as a fourth test year: trained on every target before 2025. The
2025 fold played no part in any design choice; the features, definitions,
hyperparameters and the 0.5 threshold were all fixed on 2022 to 2024.

In [9]:
run("b1_dobra_2025.py")

inc100 h=1: 2025 treinado e avaliado [36s]


inc100 h=2: 2025 treinado e avaliado [36s]


inc100 h=3: 2025 treinado e avaliado [36s]


inc100 h=4: 2025 treinado e avaliado [38s]


inc100 h=5: 2025 treinado e avaliado [37s]


inc100 h=6: 2025 treinado e avaliado [35s]


... (3 outputs left out)

canal h=4: 2025 treinado e avaliado [38s]


canal h=5: 2025 treinado e avaliado [38s]


canal h=6: 2025 treinado e avaliado [38s]


period                                                         2022-2024                  2025
definition h subset               model                                                       
canal      1 continuation_ref_tm1 climatology_corr  0.492 [0.476, 0.509]  0.710 [0.687, 0.734]
                                  rf                0.761 [0.744, 0.779]  0.640 [0.614, 0.667]
                                  xgb               0.734 [0.715, 0.754]  0.671 [0.648, 0.698]
             onset_ref_t          climatology_corr  0.504 [0.490, 0.520]  0.674 [0.650, 0.696]
                                  rf                0.727 [0.715, 0.738]  0.699 [0.679, 0.721]
                                  xgb               0.743 [0.731, 0.754]  0.718 [0.696, 0.739]
             onset_ref_tm1        climatology_corr  0.503 [0.489, 0.516]  0.699 [0.676, 0.721]
                                  rf                0.766 [0.755, 0.777]  0.744 [0.725, 0.764]
                                  xgb             

**B2.** A model trained only on months in which the municipality was calm, compared
with the single model on the same calm test months.

In [10]:
run("b2_modelo_onset.py")

h=1 ref=s_t 2022 ok


h=1 ref=s_t 2023 ok


h=1 ref=s_t 2024 ok


h=1 ref=s_tm1 2022 ok


h=1 ref=s_tm1 2023 ok


h=1 ref=s_tm1 2024 ok


... (3 outputs left out)

h=2 ref=s_tm1 2022 ok


h=2 ref=s_tm1 2023 ok


h=2 ref=s_tm1 2024 ok


                 onset_auroc  onset_auprc  recall_onset_05  false_alarm_05  precision_05  precision_at_5pct  onset_rate    min    max
h ref model                                                                                                                          
1 t   rf_onset         0.852        0.357            0.572           0.122         0.341              0.238       0.091  0.318  0.419
      rf_unico         0.836        0.337            0.463           0.101         0.349              0.286       0.091  0.300  0.391
      xgb_onset        0.858        0.363            0.443           0.073         0.380              0.250       0.091  0.358  0.366
      xgb_unico        0.855        0.356            0.432           0.075         0.373              0.287       0.091  0.341  0.370
  t-1 rf_onset         0.888        0.615            0.713           0.140         0.515              0.430       0.160  0.574  0.687
      rf_unico         0.887        0.628            0.616    

**B3.** The two relative definitions (historical range and standard score) with
thresholds computed only from years before each test year, models retrained on
those labels.

In [11]:
run("b3_limiares_passado.py")

totais estaduais de casos de dengue por ano:
ano
2015     682460
2016     177797
2017       6319
2018      14790
2019     410265
2020     195081
2021     145217
2022     331689
2023     326719
2024    2155015
2025     883472
2026      55113


2022 canal h=1 ok


2022 canal h=2 ok


2022 zscore h=1 ok


2022 zscore h=2 ok


2023 canal h=1 ok


... (4 outputs left out)

2024 canal h=2 ok


2024 zscore h=1 ok


2024 zscore h=2 ok


 test_year      epidemic_years_rule     epidemic_years_paper  median_prior_total
      2022             [2015, 2019] [2015, 2019, 2024, 2025]            177797.0
      2023       [2015, 2019, 2022] [2015, 2019, 2024, 2025]            186439.0
      2024 [2015, 2019, 2022, 2023] [2015, 2019, 2024, 2025]            195081.0
 test_year definition  prev_paper  prev_past_only  labels_changed  pct_changed  kappa
      2022      canal       0.195           0.255             469        0.061  0.827
      2022     zscore       0.153           0.242             689        0.089  0.723
      2023      canal       0.158           0.224             524        0.068  0.783
      2023     zscore       0.119           0.211             717        0.093  0.669
      2024      canal       0.443           0.498             489        0.063  0.874
      2024     zscore       0.429           0.486             529        0.068  0.863
                                         prevalence  auroc  auprc  onset_a

**B4.** A random search of 20 configurations per model, chosen on the year before
each test year, so the test year plays no part in the choice.

In [12]:
run("b4_tuning.py")

h=1 2022 rf: val AUPRC melhor 0.6397 vs paper 0.6166 [280s]


h=1 2022 xgb: val AUPRC melhor 0.6136 vs paper 0.5827 [46s]


h=1 2023 rf: val AUPRC melhor 0.8074 vs paper 0.8052 [344s]


h=1 2023 xgb: val AUPRC melhor 0.8041 vs paper 0.7970 [52s]


h=1 2024 rf: val AUPRC melhor 0.7999 vs paper 0.7956 [411s]


h=1 2024 xgb: val AUPRC melhor 0.8114 vs paper 0.8090 [61s]


... (3 outputs left out)

h=2 2023 xgb: val AUPRC melhor 0.7173 vs paper 0.7144 [51s]


h=2 2024 rf: val AUPRC melhor 0.7178 vs paper 0.7076 [398s]


h=2 2024 xgb: val AUPRC melhor 0.7413 vs paper 0.7358 [57s]


             auroc  auprc  onset_auroc_ref_t  onset_auprc_ref_t  recall_onset_ref_t  recall_cont_ref_t  false_alarm_calm_ref_t  onset_auroc_ref_tm1  onset_auprc_ref_tm1
h model                                                                                                                                                                 
1 rf         0.914  0.806              0.836              0.337               0.463              0.885                   0.101                0.887                0.628
  rf_tuned   0.916  0.812              0.839              0.334               0.320              0.771                   0.071                0.889                0.639
  xgb        0.921  0.820              0.855              0.356               0.432              0.846                   0.075                0.896                0.646
  xgb_tuned  0.921  0.818              0.851              0.352               0.496              0.883                   0.098                0.895        

**C2.** Isotonic calibration fitted on the out-of-sample predictions of the year
before each test year, and what the calibrated scores mean for a rule that acts
when the probability exceeds the ratio of the cost of acting to the loss avoided.

In [13]:
run("c2_calibracao.py")

h=1 ok


h=3 ok


 h model    version  brier  brier_skill_vs_base_rate    ece  max_abs_gap  mean_gap_0.2_to_0.8  auroc  auprc  mean_pred_calm_tm1  onset_rate_calm_tm1
 1    rf        raw 0.1063                    0.4379 0.0883       0.1981              -0.1497 0.9145 0.8063              0.2429               0.1525
 1    rf calibrated 0.1032                    0.4542 0.0396       0.1320              -0.0168 0.9138 0.7900              0.1369               0.1525
 1   xgb        raw 0.0980                    0.4818 0.0351       0.1514              -0.0965 0.9208 0.8197              0.1793               0.1525
 1   xgb calibrated 0.1111                    0.4126 0.0545       0.1403               0.0410 0.9198 0.8032              0.1275               0.1525
 1  lgbm        raw 0.0974                    0.4851 0.0387       0.1137              -0.0591 0.9187 0.8188              0.1556               0.1525
 1  lgbm calibrated 0.1113                    0.4117 0.0500       0.1244               0.0333 0.9177 0.802

In [14]:
print("Files written to", OUT.relative_to(PROJECT_ROOT))
for p in sorted(OUT.iterdir()):
    print(f"  {p.name:<40} {p.stat().st_size / 1024:8.0f} KB")

Files written to data/train/revision
  a3_auditoria_features.csv                       3 KB
  a3_onsets_ref_tm1.csv                           1 KB
  a3_recall_ref_tm1.csv                           3 KB
  a5_tabela1_ic.csv                               2 KB
  a5_tabela2_ic.csv                               4 KB
  a6_orcamento_calmos.csv                        70 KB
  a9_shap_onset.csv                               4 KB
  a9_shap_onset_por_dobra.csv                     7 KB
  b1_dobra_2025.csv                             100 KB
  b1_dobra_2025_ic.csv                          121 KB
  b1_orcamento_2025.csv                          15 KB
  b1_pred_dengue_canal_h1.parquet              1189 KB
  b1_pred_dengue_canal_h2.parquet              1188 KB
  b1_pred_dengue_canal_h3.parquet              1188 KB
  b1_pred_dengue_canal_h4.parquet              1187 KB
  b1_pred_dengue_canal_h5.parquet              1187 KB
  b1_pred_dengue_canal_h6.parquet              1187 KB
  b1_pred_dengue_inc100_h1.p